In [ ]:
import numpy as np
from scipy.fft import fft
import matplotlib.pyplot as plt
import pickle
%matplotlib inline

In [ ]:
with open('data_set_1.pkl', 'rb') as f:
    data_set_1 = pickle.load(f)

with open('data_set_2.pkl', 'rb') as f:
    data_set_2 = pickle.load(f)

xvals1 = data_set_1[:, 0]
yvals1 = data_set_1[:, 1]

xvals2 = data_set_2[:, 0]
yvals2 = data_set_2[:, 1]

In [ ]:
def xp_mat_maker(xvals, p):
    xp = np.ones((xvals.size, p+1))
    for jj in range(p):
        xp[:, jj+1] = xvals * xp[:, jj]
    return xp

In [ ]:
def cos_mat_maker(xvals, freqs):
    cp = np.ones((xvals.size, len(freqs)))
    for jj, freq in enumerate(freqs):
        cp[:, jj] = np.cos(freq * xvals)
    return cp

In [ ]:
def least_squares_solve(mat, yvals):
    u, s, vt = np.linalg.svd(mat, full_matrices=False)
    alpha = (vt.T @ np.diag(1./s) @ u.T) @ yvals.reshape(-1, 1)
    error = np.linalg.norm(mat @ alpha - yvals.reshape(-1, 1))
    return alpha, error 

**Problem 1**: Suppose I have a Gaussian random variable ${\bf X}\in \mathbb{R}^{n+m}$ described by the distribution
$$
p({\bf x}) = \frac{1}{\sqrt{(2\pi)^{n+m} \text{det}({\bf C})}} e^{-\frac{1}{2}\left<({\bf x}-\boldsymbol{\mu}), {\bf C}^{-1}({\bf x}-\boldsymbol{\mu})\right>}, ~ {\bf C} > 0.
$$
Let ${\bf x} = ({\bf x}_{1},{\bf x}_{2})^{T}$ where ${\bf x}_{1}\in\mathbb{R}^{n}$ and ${\bf x}_{2}\in\mathbb{R}^{m}$.  Find the conditional probability distribution
$$
p({\bf x}_{1}|{\bf x}_{2}) = \frac{e^{-\frac{1}{2}\left<({\bf x}-\boldsymbol{\mu}), {\bf C}^{-1}({\bf x}-\boldsymbol{\mu})\right>}}{\int_{\mathbb{R}^{n}} e^{-\frac{1}{2}\left<({\bf x}-\boldsymbol{\mu}), {\bf C}^{-1}({\bf x}-\boldsymbol{\mu})\right>} d{\bf x}_{1}}
$$

To do this, we let $\tilde{{\bf x}}={\bf x} - \boldsymbol{\mu} = (\tilde{{\bf x}}_{1} ~\tilde{{\bf x}}_{2})^{T}$.  Now if we partition the $(m+n)\times(m+n)$ matrix ${\bf C}$ as (remember ${\bf C}^{T}={\bf C}$)
$$
{\bf C} = \begin{pmatrix} {\bf C}_{nn} & {\bf C}_{nm}\\ {\bf C}^{T}_{nm} & {\bf C}_{mm}\end{pmatrix}
$$
where ${\bf C}_{nn}$ is a $n\times n$ symmetric matrix, ${\bf C}_{mm}$ is $m\times m$ symmetric matrix, and ${\bf C}_{nm}$ is $n\times m$, then what we need to figure out is how to write the inverse of ${\bf C}$ in a way that respects the partition structure.  

*Problem 1.1*: Given that ${\bf C}>0$ show that ${\bf C}_{nn}$ and ${\bf C}_{mm}$ are also positive definite.  

*Problem 1.2*: Write ${\bf C}^{-1}$ in the partitioned form
$$
{\bf C}^{-1} = \begin{pmatrix} {\bf A}_{nn} & {\bf A}_{nm}\\ {\bf A}^{T}_{nm} & {\bf A}_{mm}\end{pmatrix}
$$
Using the same arguments from above, show that ${\bf A}_{nn}$ and ${\bf A}_{mm}$ are positive definite matrices.  

*Problem 1.3* Show that 

\begin{align*}
{\bf A}_{nn}{\bf C}_{nn} + {\bf A}_{nm}{\bf C}_{nm}^{T} = & {\bf I}_{n}\\
{\bf A}_{nn}{\bf C}_{nm} + {\bf A}_{nm}{\bf C}_{mm} = & {\bf O}_{nm}\\
{\bf A}_{nm}^{T}{\bf C}_{nm} + {\bf A}_{mm}{\bf C}_{mm} = & {\bf I}_{m}
\end{align*}


*Problem 1.4* Show that 

\begin{align*}
{\bf A}_{nn} = & {\bf C}_{nn}^{-1}\left({\bf I}_{n} - {\bf C}_{nm}{\bf C}_{mm}^{-1}{\bf C}_{nm}^{T}{\bf C}_{nn}^{-1}\right)^{-1}\\
{\bf A}_{nm} = & -{\bf A}_{nn}{\bf C}_{nm}{\bf C}_{mm}^{-1}\\
{\bf A}_{mm} = & \left({\bf I}_{m} - {\bf A}_{nm}^{T}{\bf C}_{nm}\right){\bf C}_{mm}^{-1}
\end{align*}

*Problem 1.5* If we write $<\tilde{{\bf x}},{\bf C}^{-1}\tilde{{\bf x}}>$ with respect to our partition of ${\bf C}^{-1}$ we have 
$$
<\tilde{{\bf x}},{\bf C}^{-1}\tilde{{\bf x}}> = \left<\tilde{{\bf x}}_{1},{\bf A}_{nn}\tilde{{\bf x}}_{1}\right> + 2\left<\tilde{{\bf x}}_{1},{\bf A}_{nm}\tilde{{\bf x}}_{2}\right> + \left<\tilde{{\bf x}}_{2},{\bf A}_{mm}\tilde{{\bf x}}_{2}\right>.
$$
Given that we know that ${\bf A}_{nn} > 0$, that means it has a spectral decomposition so that ${\bf A}_{nn} = {\bf U}_{n}\boldsymbol{\Sigma}^{2}_{n}{\bf U}^{T}_{n}$.  Using this, transform $<\tilde{{\bf x}},{\bf C}^{-1}\tilde{{\bf x}}>$, "complete the square", and thus show 

\begin{align*}
<\tilde{{\bf x}},{\bf C}^{-1}\tilde{{\bf x}}> = & \left|\left|\boldsymbol{\Sigma}_{n}{\bf U}^{T}_{n}\tilde{{\bf x}}_{1}+\boldsymbol{\Sigma}_{n}^{-1}{\bf U}^{T}_{n}{\bf A}_{nm}\tilde{{\bf x}}_{2}\right|\right|_{2}^{2} + \left<\tilde{{\bf x}}_{2},{\bf A}_{mm}\tilde{{\bf x}}_{2}\right> -  \left|\left|\boldsymbol{\Sigma}_{n}^{-1}{\bf U}^{T}_{n}{\bf A}_{nm}\tilde{{\bf x}}_{2}\right|\right|_{2}^{2}\\
= & \left|\left|{\bf A}_{nn}\tilde{{\bf x}}_{1}+{\bf A}_{nn}^{-1}{\bf A}_{nm}\tilde{{\bf x}}_{2}\right|\right|_{2}^{2} + \left<\tilde{{\bf x}}_{2},{\bf A}_{mm}\tilde{{\bf x}}_{2}\right> -  \left|\left|{\bf A}_{nn}^{-1}{\bf A}_{nm}\tilde{{\bf x}}_{2}\right|\right|_{2}^{2}
\end{align*}

Note, you need to make use of the fact that ${\bf U}_{n}{\bf U}_{n}^{T}={\bf U}_{n}^{T}{\bf U}_{n}={\bf I}_{n}$ and $\left|\left|{\bf U}_{n}^{T}{\bf y}\right|\right|_{2} = \left|\left|{\bf y}\right|\right|_{2}$.  So for example, that is how I can write

\begin{align*}
\left|\left|\boldsymbol{\Sigma}_{n}^{-1}{\bf U}^{T}_{n}{\bf A}_{nm}\tilde{{\bf x}}_{2}\right|\right|_{2}^{2} = & \left|\left|{\bf U}_{n}^{T}{\bf U}_{n}\boldsymbol{\Sigma}_{n}^{-1}{\bf U}^{T}_{n}{\bf A}_{nm}\tilde{{\bf x}}_{2}\right|\right|_{2}^{2}\\
= & \left|\left|{\bf U}_{n}^{T}{\bf A}_{nn}^{-1}{\bf A}_{nm}\tilde{{\bf x}}_{2}\right|\right|_{2}^{2}\\
= & \left|\left|{\bf A}_{nn}^{-1}{\bf A}_{nm}\tilde{{\bf x}}_{2}\right|\right|_{2}^{2}
\end{align*}

*Problem 1.6* Having gotten through all that (COME TO OFFICE HOURS!!!!!!!!!!!!!!!), show that 
$$
p({\bf x}_{1}|{\bf x}_{2}) = \frac{\text{det}({\bf A}_{nn})\text{exp}\left(-\left|\left|{\bf A}_{nn}\tilde{{\bf x}}_{1}+{\bf A}_{nn}^{-1}{\bf A}_{nm}\tilde{{\bf x}}_{2}\right|\right|_{2}^{2}\right)}{(2\pi)^{n/2}}
$$

**Problem 2**: Using a least-squares approach, find a polynomial model of the data set represented by `xvals1` and `yvals1`, i.e. find $p$ and corresponding $\alpha_{l}$ such that 

$$
y_{j} \approx \sum_{l=0}^{p}\alpha_{l}x_{j}^{l}
$$

Please explain how you arrived at your choice for $p$.  What is the condition number of your matrix $X_{p}$?  How does this value influence your choice of $p$?  What role does regularization play in improving your results?  

In [ ]:
pval = 1  
xp = xp_mat_maker(xvals1, pval)
alpha, error = least_squares_solve(xp, yvals1)

In [ ]:
plt.plot(xvals1, xp @ alpha, color='k', ls='--')
plt.scatter(xvals1, yvals1, color='r', s = 1.)
plt.xlabel(r"$x$")
plt.ylabel(r"$y(x)$");

**Problem 3**: Using a least-squares approach, find a cos model of the data set reprsented by `xvals2` and `yvals2`.  In other finds, find $\alpha_{l}$ and $\omega_{l}$ such that

$$
y_{j} \approx \sum_{l=0}^{p}\alpha_{l}\cos(\omega_{l}x_{j}).
$$

To figure out the frequencies, you need to use a Fast Fourier Transform (FFT; see Book/Notes).  Then you find the $\alpha_{l}$ terms using least-squares fitting.  Explain your choice of frequencies and the quality of fit that this gives you to your data set.  How would you regularize your fit?  

In [ ]:
npts = yvals2.size
kvals = np.arange(int(npts/2)) * 2.*np.pi/xvals2[-1] # scale using period equal to max of xval
yfreq_plot = np.abs(fft(yvals2))[:int(npts/2)]
plt.plot(kvals, yfreq_plot) # you might want to use slicing to zoom in on the frequencies you're most interested in

In [ ]:
freqs = #what frequencies are you going to pick?
cp = cos_mat_maker(xvals2, freqs)
alpha, error = least_squares_solve(cp, yvals2)

In [ ]:
plt.plot(xvals2, cp @ alpha, color='k', ls='--')
plt.scatter(xvals2, yvals2, color='r', s = 1.)
plt.xlabel(r"$x$")
plt.ylabel(r"$y(x)$")